# LSTM Model Training for Current Candle Close Prediction

This notebook trains the LSTM model for predicting whether the current candle closes green (close > open) or red (close <= open).

**Purpose**: Train a deep learning model that Bot 1 will use for real-time candle close prediction on BTCUSDT 15m timeframe.

**Sections**:
1. Data Loading & Preprocessing
2. Feature Engineering
3. Train/Val/Test Split (Time-based)
4. Model Definition & Training
5. Evaluation & Metrics
6. Threshold Tuning per Regime
7. Model Saving & Artifacts

## 1. Setup & Imports

In [1]:
import sys
import os
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Add parent directory to path to import modules
sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, brier_score_loss
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import json

# Import custom modules
from features import FeatureEngine
from regime_detection import RegimeDetector, RegimeType
from model_inference import CandleLSTM

print("Imports successful!")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

Imports successful!
PyTorch version: 2.1.2+cpu
CUDA available: False


## 2. Data Loading

Load historical candles data. You'll need to either:
- Use existing `data/candles_15m.csv` from live bot
- Or download historical data from Binance API

In [2]:
# Option 1: Load from existing CSV
data_path = Path("../data/candles_15m.csv")

if data_path.exists():
    df = pd.read_csv(data_path)
    print(f"Loaded {len(df)} candles from {data_path}")
else:
    print("No data file found. Creating sample data...")
    # Create sample data for demonstration
    np.random.seed(42)
    n_candles = 5000
    
    base_price = 50000
    prices = []
    current_price = base_price
    
    for i in range(n_candles):
        change = np.random.randn() * 100
        current_price += change
        open_price = current_price
        high_price = open_price + abs(np.random.randn() * 50)
        low_price = open_price - abs(np.random.randn() * 50)
        close_price = open_price + np.random.randn() * 80
        volume = 100 + abs(np.random.randn() * 50)
        
        prices.append({
            'symbol': 'BTCUSDT',
            'interval': '15m',
            'open_time_ms': 1700000000000 + i * 900000,
            'close_time_ms': 1700000000000 + (i + 1) * 900000,
            'open': open_price,
            'high': high_price,
            'low': low_price,
            'close': close_price,
            'volume': volume,
        })
    
    df = pd.DataFrame(prices)
    df.to_csv(data_path, index=False)
    print(f"Created sample data: {len(df)} candles")

# Display basic info
print(f"\nDataFrame shape: {df.shape}")
print(f"\nFirst candle: {pd.to_datetime(df['open_time_ms'].iloc[0], unit='ms')}")
print(f"Last candle: {pd.to_datetime(df['open_time_ms'].iloc[-1], unit='ms')}")
print(f"\nColumns: {df.columns.tolist()}")
df.head()

Loaded 105200 candles from ..\data\candles_15m.csv

DataFrame shape: (105200, 13)

First candle: 2022-12-29 19:00:00
Last candle: 2025-12-29 16:00:00

Columns: ['symbol', 'interval', 'open_time_ms', 'close_time_ms', 'open', 'high', 'low', 'close', 'volume', 'num_trades', 'quote_volume', 'taker_buy_base_vol', 'taker_buy_quote_vol']


,symbol,interval,open_time_ms,close_time_ms,open,high,low,close,volume,num_trades,quote_volume,taker_buy_base_vol,taker_buy_quote_vol
0,BTCUSDT,15m,1672340400000,1672341299999,16633.73,16642.08,16632.19,16636.81,1473.22824,39562,2.451041e+07,708.30915,1.178443e+07
1,BTCUSDT,15m,1672341300000,1672342199999,16636.44,16638.30,16603.02,16611.40,2301.06899,52101,3.824412e+07,1116.78242,1.856101e+07
2,BTCUSDT,15m,1672342200000,1672343099999,16610.72,16624.88,16608.93,16611.14,1399.19194,38756,2.324834e+07,707.57306,1.175692e+07
3,BTCUSDT,15m,1672343100000,1672343999999,16611.14,16617.59,16590.82,16612.40,1680.50013,46523,2.790837e+07,829.02384,1.376806e+07
4,BTCUSDT,15m,1672344000000,1672344899999,16612.40,16618.63,16597.67,16609.42,1547.92314,42187,2.571085e+07,785.08347,1.304040e+07


## 3. Feature Engineering

Build features using our FeatureEngine class.

**IMPORTANT FIX (2026-01-02):**
- Changed to predict NEXT candle (not current candle)
- Eliminates data leakage from `is_bullish` and other features
- Features from candles [i-63 to i] → Predict candle [i+1]
- More useful for trading: predicts future, not past

In [ ]:
# Create labels (1 if green candle, 0 if red)
# NOTE: This is computed for convenience but build_sequences() will use NEXT candle's label
df['label'] = (df['close'] > df['open']).astype(int)

print(f"Label distribution:")
print(df['label'].value_counts())
print(f"\nGreen candles: {df['label'].sum() / len(df) * 100:.1f}%")
print(f"\nNOTE: Model will predict NEXT candle (not current) to avoid data leakage")

Label distribution:
label
1    52817
0    52383
Name: count, dtype: int64

Green candles: 50.2%


In [ ]:
def build_sequences(df, sequence_length=64):
    """
    Build sequences for LSTM training.
    
    FIXED: Now predicts the NEXT candle (not current) to eliminate data leakage.
    Features from candles [i-63 to i] predict candle [i+1].
    
    Returns X (features), y (labels), and metadata.
    """
    engine = FeatureEngine(buffer_size=sequence_length)
    
    X_list = []
    y_list = []
    timestamps = []
    
    # Use range instead of iterrows, stop at len(df)-1 to have room for next candle
    for idx in range(len(df) - 1):
        # Add current candle to buffer
        candle = df.iloc[idx].to_dict()
        engine.add_candle(candle)
        
        # Once we have enough candles, build features
        if engine.is_ready():
            features = engine.build_features()
            
            # CRITICAL FIX: Label is from NEXT candle (idx + 1), not current
            next_candle = df.iloc[idx + 1]
            next_label = int(next_candle['close'] > next_candle['open'])
            
            if features is not None:
                X_list.append(features)
                y_list.append(next_label)  # Predict NEXT candle
                timestamps.append(next_candle['open_time_ms'])  # Next candle timestamp
        
        # Progress indicator
        if idx % 1000 == 0:
            print(f"Processed {idx}/{len(df)-1} candles...")
    
    X = np.array(X_list)
    y = np.array(y_list)
    timestamps = np.array(timestamps)
    
    print(f"\n✓ FIXED: Predicting NEXT candle (no data leakage)")
    print(f"  Features from 64 candles → Label from candle 65")
    
    return X, y, timestamps

print("Building feature sequences (this may take a few minutes)...")
X, y, timestamps = build_sequences(df, sequence_length=64)

print(f"\nFeature matrix shape: {X.shape}")
print(f"Labels shape: {y.shape}")
print(f"Features per candle: {X.shape[2]}")

Building feature sequences (this may take a few minutes)...
Processed 0/105200 candles...
Processed 1000/105200 candles...
Processed 2000/105200 candles...
Processed 3000/105200 candles...
Processed 4000/105200 candles...
Processed 5000/105200 candles...
Processed 6000/105200 candles...
Processed 7000/105200 candles...
Processed 8000/105200 candles...
Processed 9000/105200 candles...
Processed 10000/105200 candles...
Processed 11000/105200 candles...
Processed 12000/105200 candles...
Processed 13000/105200 candles...
Processed 14000/105200 candles...
Processed 15000/105200 candles...
Processed 16000/105200 candles...
Processed 17000/105200 candles...
Processed 18000/105200 candles...
Processed 19000/105200 candles...
Processed 20000/105200 candles...
Processed 21000/105200 candles...
Processed 22000/105200 candles...
Processed 23000/105200 candles...
Processed 24000/105200 candles...
Processed 25000/105200 candles...
Processed 26000/105200 candles...
Processed 27000/105200 candles...
P

## 4. Train/Val/Test Split

**CRITICAL**: Must use time-based splitting (NOT random) to avoid data leakage!

In [ ]:
# Time-based split: 70% train, 15% val, 15% test
n_samples = len(X)
train_end = int(n_samples * 0.70)
val_end = int(n_samples * 0.85)

X_train, y_train = X[:train_end], y[:train_end]
X_val, y_val = X[train_end:val_end], y[train_end:val_end]
X_test, y_test = X[val_end:], y[val_end:]

print("Dataset splits:")
print(f"Train: {len(X_train)} samples ({len(X_train)/n_samples*100:.1f}%)")
print(f"Val:   {len(X_val)} samples ({len(X_val)/n_samples*100:.1f}%)")
print(f"Test:  {len(X_test)} samples ({len(X_test)/n_samples*100:.1f}%)")

print(f"\nTrain label distribution:")
print(f"  Green: {y_train.sum()} ({y_train.mean()*100:.1f}%)")
print(f"  Red:   {len(y_train) - y_train.sum()} ({(1-y_train.mean())*100:.1f}%)")

## 5. PyTorch Dataset & DataLoader

In [ ]:
class CandleDataset(Dataset):
    """PyTorch Dataset for candle sequences"""
    
    def __init__(self, X, y):
        self.X = torch.from_numpy(X).float()
        self.y = torch.from_numpy(y).float().unsqueeze(1)
    
    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

# Create datasets
train_dataset = CandleDataset(X_train, y_train)
val_dataset = CandleDataset(X_val, y_val)
test_dataset = CandleDataset(X_test, y_test)

# Create dataloaders
batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print(f"DataLoaders created with batch_size={batch_size}")
print(f"Train batches: {len(train_loader)}")
print(f"Val batches: {len(val_loader)}")
print(f"Test batches: {len(test_loader)}")

## 6. Model Training

In [ ]:
# Model hyperparameters
input_size = X.shape[2]  # Number of features
hidden_size = 128
num_layers = 2
dropout = 0.3
learning_rate = 0.001
num_epochs = 50

# Device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Training on: {device}")

# Initialize model
model = CandleLSTM(
    input_size=input_size,
    hidden_size=hidden_size,
    num_layers=num_layers,
    dropout=dropout
)
model.to(device)

# Loss and optimizer
criterion = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

# Learning rate scheduler
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=5, verbose=True
)

print(f"\nModel architecture:")
print(model)
print(f"\nTotal parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
# Training loop
train_losses = []
val_losses = []
val_accuracies = []
best_val_loss = float('inf')
patience_counter = 0
early_stop_patience = 10

for epoch in range(num_epochs):
    # Training phase
    model.train()
    train_loss = 0.0
    
    for batch_X, batch_y in train_loader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)
        
        # Forward pass
        outputs = model(batch_X)
        loss = criterion(outputs, batch_y)
        
        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item()
    
    train_loss /= len(train_loader)
    train_losses.append(train_loss)
    
    # Validation phase
    model.eval()
    val_loss = 0.0
    val_preds = []
    val_labels = []
    
    with torch.no_grad():
        for batch_X, batch_y in val_loader:
            batch_X, batch_y = batch_X.to(device), batch_y.to(device)
            
            outputs = model(batch_X)
            loss = criterion(outputs, batch_y)
            val_loss += loss.item()
            
            val_preds.extend(outputs.cpu().numpy())
            val_labels.extend(batch_y.cpu().numpy())
    
    val_loss /= len(val_loader)
    val_losses.append(val_loss)
    
    # Calculate accuracy
    val_preds_binary = (np.array(val_preds) > 0.5).astype(int)
    val_accuracy = accuracy_score(val_labels, val_preds_binary)
    val_accuracies.append(val_accuracy)
    
    # Learning rate scheduling
    scheduler.step(val_loss)
    
    # Print progress
    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f"Epoch [{epoch+1}/{num_epochs}] "
              f"Train Loss: {train_loss:.4f} | "
              f"Val Loss: {val_loss:.4f} | "
              f"Val Acc: {val_accuracy:.4f}")
    
    # Save best model
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_counter = 0
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_loss': val_loss,
            'val_accuracy': val_accuracy,
        }, '../models/current_candle_lstm.pt')
        print(f"  ✓ New best model saved (val_loss: {val_loss:.4f})")
    else:
        patience_counter += 1
    
    # Early stopping
    if patience_counter >= early_stop_patience:
        print(f"\nEarly stopping triggered at epoch {epoch+1}")
        break

print(f"\nTraining complete!")
print(f"Best validation loss: {best_val_loss:.4f}")

## 7. Visualization - Training Progress

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Plot losses
axes[0].plot(train_losses, label='Train Loss', linewidth=2)
axes[0].plot(val_losses, label='Val Loss', linewidth=2)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training and Validation Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Plot accuracy
axes[1].plot(val_accuracies, label='Val Accuracy', linewidth=2, color='green')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('Validation Accuracy')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 8. Model Evaluation on Test Set

In [ ]:
# Load best model
checkpoint = torch.load('../models/current_candle_lstm.pt', map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

# Get predictions on test set
test_preds = []
test_probs = []
test_labels = []

with torch.no_grad():
    for batch_X, batch_y in test_loader:
        batch_X = batch_X.to(device)
        outputs = model(batch_X)
        
        test_probs.extend(outputs.cpu().numpy())
        test_preds.extend((outputs > 0.5).cpu().numpy())
        test_labels.extend(batch_y.numpy())

test_probs = np.array(test_probs).flatten()
test_preds = np.array(test_preds).flatten().astype(int)
test_labels = np.array(test_labels).flatten().astype(int)

# Calculate metrics
accuracy = accuracy_score(test_labels, test_preds)
precision = precision_score(test_labels, test_preds)
recall = recall_score(test_labels, test_preds)
f1 = f1_score(test_labels, test_preds)
auc = roc_auc_score(test_labels, test_probs)
brier = brier_score_loss(test_labels, test_probs)

print("=" * 60)
print("TEST SET EVALUATION")
print("=" * 60)
print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1 Score:  {f1:.4f}")
print(f"AUC-ROC:   {auc:.4f}")
print(f"Brier Score: {brier:.4f} (lower is better)")
print("=" * 60)

In [ ]:
# Confusion Matrix
cm = confusion_matrix(test_labels, test_preds)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Red', 'Green'], yticklabels=['Red', 'Green'])
plt.title('Confusion Matrix on Test Set')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.show()

print(f"\nConfusion Matrix:")
print(f"True Negatives (Red predicted as Red): {cm[0,0]}")
print(f"False Positives (Red predicted as Green): {cm[0,1]}")
print(f"False Negatives (Green predicted as Red): {cm[1,0]}")
print(f"True Positives (Green predicted as Green): {cm[1,1]}")

## 9. Calibration Analysis

Check if the predicted probabilities are well-calibrated

In [ ]:
# Calibration curve (reliability diagram)
def compute_calibration_curve(y_true, y_prob, n_bins=10):
    """Compute calibration curve."""
    bins = np.linspace(0, 1, n_bins + 1)
    bin_centers = (bins[:-1] + bins[1:]) / 2
    
    bin_counts = np.zeros(n_bins)
    bin_true_probs = np.zeros(n_bins)
    
    for i in range(n_bins):
        mask = (y_prob >= bins[i]) & (y_prob < bins[i+1])
        if mask.sum() > 0:
            bin_counts[i] = mask.sum()
            bin_true_probs[i] = y_true[mask].mean()
        else:
            bin_true_probs[i] = np.nan
    
    return bin_centers, bin_true_probs, bin_counts

bin_centers, bin_true_probs, bin_counts = compute_calibration_curve(test_labels, test_probs, n_bins=10)

plt.figure(figsize=(10, 6))
plt.plot([0, 1], [0, 1], 'k--', label='Perfect Calibration', linewidth=2)
plt.plot(bin_centers, bin_true_probs, 'bo-', label='Model', linewidth=2, markersize=8)
plt.xlabel('Predicted Probability')
plt.ylabel('True Probability')
plt.title('Calibration Curve (Reliability Diagram)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.xlim([0, 1])
plt.ylim([0, 1])
plt.show()

print(f"\nCalibration Info:")
print(f"Brier Score: {brier:.4f} (measures calibration + sharpness)")
print(f"Perfect calibration = predicted probabilities match true frequencies")

## 10. Threshold Tuning

Find optimal thresholds for different regimes to maximize precision while maintaining recall

In [ ]:
def evaluate_threshold(y_true, y_prob, threshold):
    """Evaluate metrics at a specific threshold."""
    y_pred = (y_prob > threshold).astype(int)
    
    if y_pred.sum() == 0:  # No predictions
        return {
            'threshold': threshold,
            'precision': 0,
            'recall': 0,
            'f1': 0,
            'alert_rate': 0
        }
    
    precision = precision_score(y_true, y_pred, zero_division=0)
    recall = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    alert_rate = y_pred.sum() / len(y_pred)
    
    return {
        'threshold': threshold,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'alert_rate': alert_rate
    }

# Test different thresholds
thresholds = np.arange(0.5, 0.90, 0.02)
results = []

for thresh in thresholds:
    result = evaluate_threshold(test_labels, test_probs, thresh)
    results.append(result)

results_df = pd.DataFrame(results)

# Plot threshold analysis
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

axes[0, 0].plot(results_df['threshold'], results_df['precision'], 'b-', linewidth=2)
axes[0, 0].set_xlabel('Threshold')
axes[0, 0].set_ylabel('Precision')
axes[0, 0].set_title('Precision vs Threshold')
axes[0, 0].grid(True, alpha=0.3)
axes[0, 0].axhline(y=0.65, color='r', linestyle='--', label='Target: 0.65')
axes[0, 0].legend()

axes[0, 1].plot(results_df['threshold'], results_df['recall'], 'g-', linewidth=2)
axes[0, 1].set_xlabel('Threshold')
axes[0, 1].set_ylabel('Recall')
axes[0, 1].set_title('Recall vs Threshold')
axes[0, 1].grid(True, alpha=0.3)

axes[1, 0].plot(results_df['threshold'], results_df['f1'], 'purple', linewidth=2)
axes[1, 0].set_xlabel('Threshold')
axes[1, 0].set_ylabel('F1 Score')
axes[1, 0].set_title('F1 Score vs Threshold')
axes[1, 0].grid(True, alpha=0.3)

axes[1, 1].plot(results_df['threshold'], results_df['alert_rate'] * 100, 'orange', linewidth=2)
axes[1, 1].set_xlabel('Threshold')
axes[1, 1].set_ylabel('Alert Rate (%)')
axes[1, 1].set_title('Alert Rate vs Threshold')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Show top thresholds by precision
print("\nTop 5 thresholds by Precision:")
print(results_df.nlargest(5, 'precision')[['threshold', 'precision', 'recall', 'alert_rate']])

## 11. Save Regime-Specific Thresholds

Create threshold configuration for different market regimes

In [ ]:
# Define regime-specific thresholds based on analysis
regime_thresholds = {
    'TREND': {
        'bull': 0.65,
        'bear': 0.65,
        'description': 'Normal trending market - base thresholds'
    },
    'CHOP': {
        'bull': 0.75,
        'bear': 0.75,
        'description': 'Choppy sideways market - higher thresholds to reduce noise'
    },
    'HIGH_VOL_TREND': {
        'bull': 0.62,
        'bear': 0.62,
        'description': 'High volatility trending - slightly lower to catch moves'
    },
    'HIGH_VOL_CHOP': {
        'bull': 0.78,
        'bear': 0.78,
        'description': 'High volatility chop - highest thresholds, very conservative'
    },
    'UNKNOWN': {
        'bull': 0.70,
        'bear': 0.70,
        'description': 'Unknown/uncertain regime - moderately conservative'
    }
}

# Save to artifacts folder
artifacts_dir = Path('../artifacts')
artifacts_dir.mkdir(exist_ok=True)

thresholds_path = artifacts_dir / 'thresholds.json'
with open(thresholds_path, 'w') as f:
    json.dump(regime_thresholds, f, indent=2)

print(f"Regime thresholds saved to: {thresholds_path}")
print("\nThresholds:")
print(json.dumps(regime_thresholds, indent=2))

## 12. Save Training Metadata

In [ ]:
# Create training run metadata
training_metadata = {
    'run_id': f"train_{datetime.now().strftime('%Y%m%d_%H%M%S')}",
    'timestamp': datetime.now().isoformat(),
    'model_type': 'LSTM',
    'model_params': {
        'input_size': input_size,
        'hidden_size': hidden_size,
        'num_layers': num_layers,
        'dropout': dropout,
    },
    'training_params': {
        'learning_rate': learning_rate,
        'batch_size': batch_size,
        'num_epochs': len(train_losses),
        'optimizer': 'Adam',
        'loss_function': 'BCELoss',
    },
    'data_info': {
        'total_samples': len(X),
        'train_samples': len(X_train),
        'val_samples': len(X_val),
        'test_samples': len(X_test),
        'sequence_length': 64,
        'num_features': input_size,
    },
    'metrics': {
        'test_accuracy': float(accuracy),
        'test_precision': float(precision),
        'test_recall': float(recall),
        'test_f1': float(f1),
        'test_auc': float(auc),
        'test_brier_score': float(brier),
        'best_val_loss': float(best_val_loss),
    },
    'thresholds': regime_thresholds,
    'feature_version': 'v3',
}

# Save to JSONL file
training_runs_path = Path('../data/training_runs.jsonl')
with open(training_runs_path, 'a') as f:
    f.write(json.dumps(training_metadata) + '\n')

print(f"Training metadata saved to: {training_runs_path}")
print(f"\nRun ID: {training_metadata['run_id']}")
print(f"Test Accuracy: {accuracy:.4f}")
print(f"Test Precision: {precision:.4f}")

## 13. Model Deployment Checklist

✅ **Model Training Complete!**

Before deploying to production:

1. ✓ Model saved to `../models/current_candle_lstm.pt`
2. ✓ Thresholds saved to `../artifacts/thresholds.json`
3. ✓ Training metadata saved to `../data/training_runs.jsonl`
4. ⚠️ Copy `.env.template` to `.env` and fill in credentials
5. ⚠️ Test model inference with `model_inference.py`
6. ⚠️ Run bot in test mode before 24/7 deployment

**Next Steps**:
- Test the bot locally: `python run_bot.py`
- Monitor logs in `logs/inference.log`
- Verify Telegram alerts are working
- Set up Windows Task Scheduler for 24/7 operation